# Блокнот для обучения Qwen3 VL 8B
**Большая часть кодов была взята из официального Colab-блокнота Unsloth, посвященного обучению Qwen3 VL 8B**

***Daniel Han, Michael Han and Unsloth team,
http://github.com/unslothai/unsloth,
2023***

***Версия кода — 2.0***

***4-я глобальная переработка; 7-я переработка данной версии***

## Принцип последовательного обучения без забывания

При обучении LoRA поверх старого адаптера, он перезаписывается — модель «забывает» то, что она только что выучила.

После каждой стадии необходимо **сливать LoRA в базу** и сохранять **полный чекпоинт (merged model)**. На следующей стадии загружать этот merged model как новую базу и вешать **новый, пустой** LoRA — так знания text остаются в базе, а LoRA учит только math (и так далее).

1. text: база = `unsloth/Qwen3-VL-8B-...` → обучение LoRA → слив в базу → сохранение в `merged_after_text`.
2. math: база = `merged_after_text` → новый LoRA → обучение → слив → сохранение в `merged_after_math`.
3. vision: база = `merged_after_math` → новый LoRA → обучаение → слив → сохранение и т.д.

В связи с этим для полного обучения на всех стадиях необходимо несколько раз менять значения в коде и перезапускать его (на момент написания кода иные варианты не обнаружены).

## Установка библиотек
А также ранняя инициализация unsloth для предотвращения ошибок

In [ ]:
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==3.6.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.57.1
!pip install trl==0.22.2
!pip install zstandard
!pip install jsonlines

In [ ]:
from unsloth import FastVisionModel

## Распаковка merged модели из предыдущей стадии
*Требуется только для math и vision*

In [ ]:
from google.colab import drive
import zipfile
drive.mount("/content/drive")

**Необходимо менять пути к файл соответственно пути хранения файлов на диске**

In [ ]:
with zipfile.ZipFile('/content/drive/MyDrive/after_vision.zip', 'r') as z: # путь/до/архива/на/диске
  z.extractall("/content/after-vision") # путь распаковки модели

In [ ]:
drive.flush_and_unmount() # отключение диска (а почему бы и нет)

## Аргументы для загрузки датасетов \ модели, обучения

In [ ]:
from typing import Dict, List, Optional, Any
import os

class Args():
    outdir = "/kaggle/working/" # унаследовано из Kaggle :p
    val = 0.02

    # Путь к модели для определенной стадии
    original_base = "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit"
    merged_after_text = "/content/after-text"
    merged_after_math = "/content/after-math"

    # Лимиты
    limit_text = 20000
    limit_math = 20000
    limit_vision = 10000
    max_length = 2048

    # Эпохи обучения (стоит на 1.0, так как обучение идет по шагам)
    epochs_text = 1.0
    epochs_math = 1.0
    epochs_vision = 1.0

    # Размеры батча
    batch_train = 5
    batch_eval = 5
    grad_accum = 6

    # Параметры логирования
    eval_steps = 10
    save_steps = 20
    logging_steps = 10

    lr = 4e-5 # 1e-4 возможный вариант
    weight_decay = 0.01
    num_workers = 6
    dataloader_pin_memory = True

    # Варианты: "adamw_8bit", "adamw_torch", "paged_adamw_8bit"
    optim = "paged_adamw_8bit"
    use_gradient_checkpointing = True

    # МЕНЯТЬ СТАДИЮ
    stage = "vision" # "text" | "math" | "vision"

args = Args()

## Оптимизация
*Запуск не обязателен, но желателен*

In [ ]:
import os
import torch

# TF32 ускоряет matmul на Ampere+ (A100 и новее)
if torch.cuda.is_available():
    major, _ = torch.cuda.get_device_capability(0)
    allow_tf32 = major >= 8
else:
    allow_tf32 = False

torch.backends.cuda.matmul.allow_tf32 = allow_tf32
torch.backends.cudnn.allow_tf32 = allow_tf32

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:512"
os.environ["CUDA_LAUNCH_BLOCKING"] = "0"  # Асинхронное выполнение

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Не выводит ошибки токенизатора

os.environ["DS_SKIP_CUDA_CHECK"] = "0"  # Проверка CUDA на всякий

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

torch.cuda.empty_cache()  # Очистка кэша перед обучением

print("Доп. Оптимизация")
print(f"  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"  CUDA Доступна: {torch.cuda.is_available()}")
print(f"  TF32 Включен: {allow_tf32}")
if torch.cuda.is_available():
    print(f"  GPU Память: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Подготовка данных
*Самый сложный этап, так как требует именно самостоятельно придуманного кода. А с этим часто проблемы.*

### Функции загрузки датасетов

In [ ]:
from datasets import load_dataset, concatenate_datasets, Dataset

# -------------- Loaders: TEXT --------------

def load_alpaca_cleaned(limit: Optional[int], num_workers) -> Dataset:
    split_expr = "train" if not limit else f"train[:{limit}]"
    ds = load_dataset("d0rj/alpaca-cleaned-ru", num_proc=num_workers, split=split_expr)

    def _convert(sample):
        problem = f"{sample['instruction']}: {sample['input']}" if sample["input"] not in (None, "") else sample["instruction"]
        solution = sample["output"]
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": problem},
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": solution},
                ],
            },
        ]
        return {"messages": messages}

    converted_ds = ds.map(_convert, remove_columns=ds.column_names, num_proc=num_workers)
    return converted_ds


def load_bactrian_x(limit: Optional[int], num_workers) -> Dataset:
    split_expr = "train" if not limit else f"train[:{limit}]"
    ds = load_dataset("MBZUAI/Bactrian-X", "ru", num_proc=num_workers, split=split_expr)

    def _convert(sample):
        problem = f"{sample['instruction']}: {sample['input']}" if sample["input"] not in (None, "") else sample["instruction"]
        solution = sample["output"]
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": problem},
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": solution},
                ],
            },
        ]
        return {"messages": messages}

    converted_ds = ds.map(_convert, remove_columns=ds.column_names, num_proc=num_workers)
    return converted_ds


def load_ru_turbo_alpaca(limit: Optional[int], num_workers) -> Dataset:
    ds = load_dataset("IlyaGusev/ru_turbo_alpaca", num_proc=num_workers, trust_remote_code=True)
    train_ds = ds["train"]
    if limit:
        train_ds = train_ds.select(range(min(limit, len(train_ds))))

    def _convert(sample):
        problem = f"{sample['instruction']}: {sample['input']}" if sample["input"] not in (None, "") else sample["instruction"]
        solution = sample["output"]
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": problem},
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": solution},
                ],
            },
        ]
        return {"messages": messages}

    converted_ds = train_ds.map(_convert, remove_columns=train_ds.column_names, num_proc=num_workers)
    return converted_ds


def build_text_stage(limit: Optional[int], num_workers) -> Dataset:
    parts: List[Dataset] = []
    for loader in (load_alpaca_cleaned, load_bactrian_x, load_ru_turbo_alpaca):
        try:
            parts.append(loader(limit=limit, num_workers=num_workers))
        except Exception as e:
            print(f"При загрузке {loader} возникла ошибка: {e}")
    if not parts:
        raise RuntimeError(f"Не удалось загрузить ни один датасет для стадии '{args.stage}'")
    return concatenate_datasets(parts)


# -------------- Loaders: MATH (TEXT) --------------

def load_ru_alpaca_math(limit: Optional[int], num_workers) -> Dataset:
    split_expr = "train" if not limit else f"train[:{limit}]"
    ds = load_dataset("ai-bond/ru-alpaca-math", split=split_expr, num_proc=num_workers)

    def _convert(sample):
        problem = f"{sample['instruction']}: {sample['input']}" if sample["input"] not in (None, "") else sample["instruction"]
        solution = sample["output"]
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": problem},
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": solution},
                ],
            },
        ]
        return {"messages": messages}

    converted_ds = ds.map(_convert, remove_columns=ds.column_names, num_proc=num_workers)
    return converted_ds


def load_gsm8k_ru(limit: Optional[int], num_workers) -> Dataset:
    split_expr = "train" if not limit else f"train[:{limit}]"
    ds = load_dataset("d0rj/gsm8k-ru", split=split_expr, num_proc=num_workers)

    def _convert(sample):
        problem = sample["question"]
        solution = sample["answer"]
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": problem},
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": solution},
                ],
            },
        ]
        return {"messages": messages}

    converted_ds = ds.map(_convert, remove_columns=ds.column_names, num_proc=num_workers)
    return converted_ds


def build_math_stage(limit: Optional[int], num_workers) -> Dataset:
    parts: List[Dataset] = []
    for loader in (load_ru_alpaca_math, load_gsm8k_ru):
        try:
            parts.append(loader(limit=limit, num_workers=num_workers))
        except Exception as e:
            print(f"При загрузке {loader} возникла ошибка: {e}")
    if not parts:
        raise RuntimeError(f"Не удалось загрузить ни один датасет для стадии '{args.stage}'")
    return concatenate_datasets(parts)


# -------------- Loaders: VISION (MATH IMAGES) --------------

def load_chartqa(limit: Optional[int], num_workers) -> Dataset:
    split_expr = "train" if not limit else f"train[:{limit}]"
    ds = load_dataset("HuggingFaceM4/ChartQA", split=split_expr, num_proc=num_workers)

    def _convert(batch):
        messages = []
        for query, image, label in zip(batch["query"], batch["image"], batch["label"]):
            messages.append(
                [
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": query},
                            {"type": "image", "image": image},
                        ],
                    },
                    {
                        "role": "assistant",
                        "content": [
                            {"type": "text", "text": label[0]},
                        ],
                    },
                ]
            )
        return {"messages": messages}

    converted_ds = ds.map(
        _convert,
        batched=True,
        remove_columns=ds.column_names,
        num_proc=num_workers,
    )
    return converted_ds


def load_aida_calculus(limit: Optional[int], num_workers) -> Dataset:
    split_expr = "train" if not limit else f"train[:{limit}]"
    ds = load_dataset("deepcopy/Aida-Calculus-Math-Handwriting", split=split_expr, num_proc=num_workers)

    def _convert(sample):
        instruction = "Write the LaTeX representation for this image."
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": instruction},
                    {"type": "image", "image": sample["image"]},
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": sample["latex"]},
                ],
            },
        ]
        return {"messages": messages}

    converted_ds = ds.map(
        _convert,
        batched=False,
        remove_columns=ds.column_names,
        num_proc=num_workers,
    )
    return converted_ds


def load_geometry3k(limit: Optional[int], num_workers) -> Dataset:
    split_expr = "test" if not limit else f"test[:{limit}]"
    ds = load_dataset("hiyouga/geometry3k", split=split_expr, num_proc=num_workers)

    def _convert(sample):
        img = sample["images"][0] if isinstance(sample["images"], list) else sample["images"]
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": sample["problem"]},
                    {"type": "image", "image": img},
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": sample["answer"]},
                ],
            },
        ]
        return {"messages": messages}

    converted_ds = ds.map(
        _convert,
        batched=False,
        remove_columns=ds.column_names,
        num_proc=num_workers,
    )
    return converted_ds


def load_latex_ocr(limit: Optional[int], num_workers) -> Dataset:
    split_expr = "train" if not limit else f"train[:{limit}]"
    ds = load_dataset("linxy/LaTeX_OCR", "full", split=split_expr, num_proc=num_workers)

    def _convert(sample):
        if sample["text"] not in (None, ""):
            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": "Write LaTeX representation of text on the following image."},
                        {"type": "image", "image": sample["image"]},
                        ],
                },
                {
                    "role": "assistant",
                    "content": [
                        {"type": "text", "text": sample["text"]},
                    ],
                },
            ]
            return {"messages": messages}
    
    converted_ds = ds.map(_convert, batched=False, remove_columns=ds.column_names, num_proc=num_workers)
    return converted_ds



def build_vision_stage(limit: Optional[int], num_workers) -> Dataset:
    parts: List[Dataset] = []
    for loader in (load_chartqa, load_aida_calculus, load_geometry3k, load_latex_ocr):
        try:
            parts.append(loader(limit=limit, num_workers=num_workers))
        except Exception as e:
            print(f"При загрузке {loader} возникла ошибка: {e}")
    if not parts:
        raise RuntimeError(f"Не удалось загрузить ни один датасет для стадии '{args.stage}'")
    return concatenate_datasets(parts)

## Загрузка датасетов, модели и обучение

In [ ]:
epochs_map = {
    "text": args.epochs_text,
    "math": args.epochs_math,
    "vision": args.epochs_vision
}
current_epochs = epochs_map.get(args.stage)
if current_epochs == None:
    raise RuntimeError("Стадия не существует, проверьте правильность в Args")

path_map = {
    "text": args.original_base,
    "math": args.merged_after_text,
    "vision": args.merged_after_math
}
base_path = path_map.get(args.stage)
if base_path == None:
    raise RuntimeError("Стадия не существует, проверьте правильность в Args")
print(f"Загрузка базы для стадии '{args.stage}': {base_path}")

In [ ]:
import torch
from unsloth.trainer import UnslothVisionDataCollator
from trl.trainer.sft_trainer import SFTTrainer # from trl
from trl.trainer.sft_config import SFTConfig # from trl

bf16_supported = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

training_args = SFTConfig(
    per_device_train_batch_size=args.batch_train,
    per_device_eval_batch_size=args.batch_eval,
    gradient_accumulation_steps=args.grad_accum,
    fp16=not bf16_supported,
    fp16_full_eval=not bf16_supported,
    warmup_ratio=0.03,  # 3% warmup
    num_train_epochs=current_epochs,
    save_strategy="steps",
    save_steps=args.save_steps,
    eval_steps=args.eval_steps,
    eval_strategy="steps",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    learning_rate=args.lr,
    logging_steps=args.logging_steps,
    bf16=bf16_supported,
    bf16_full_eval=bf16_supported,
    tf32=True,
    optim=args.optim,
    weight_decay=args.weight_decay,
    lr_scheduler_type="cosine",
    seed=3407,
    output_dir=args.outdir,
    report_to="none",
    gradient_checkpointing=args.use_gradient_checkpointing,

    # Performance optimizations
    dataloader_pin_memory=args.dataloader_pin_memory,
    dataloader_num_workers=args.num_workers,


    # ОБЯЗАТЕЛЬНО для обучения на изображениях:
    remove_unused_columns=False,
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
    max_length=args.max_length,

    # AttributeError: 'int' object has no attribute 'mean'
    # Происходит когда внутренняя функция возвращает число вместо тензора (баг unsloth)
    # False исправляет проблему
    average_tokens_across_devices=False,
)

In [ ]:
import torch

def create_model_and_tokenizer():
    model, tokenizer = FastVisionModel.from_pretrained(
        base_path,
        max_seq_length=args.max_length,
        load_in_4bit=True,
        use_gradient_checkpointing="unsloth" if args.use_gradient_checkpointing else False,
    )

    model = FastVisionModel.get_peft_model(
        model,
        finetune_vision_layers     = True if args.stage == "vision" else False, # False if not finetuning vision layers
        finetune_language_layers   = True, # False if not finetuning language layers
        finetune_attention_modules = True, # False if not finetuning attention layers
        finetune_mlp_modules       = True, # False if not finetuning MLP layers

        r = 16,           # The larger, the higher the accuracy, but might overfit
        lora_alpha = 16,  # Recommended alpha == r at least
        lora_dropout = 0,
        bias = "none",
        random_state = 3407, # 42 — ответ на вопрос жизни, вселенной и вообще?
        use_rslora = False,  # Unsloth supports rank stabilized LoRA
        loftq_config = None, # And LoftQ
    )
    return model, tokenizer

model, tokenizer = create_model_and_tokenizer()

In [ ]:
if args.stage == "text":  # ЕСЛИ СТАДИЯ == text
    ds = build_text_stage(limit=args.limit_text, num_workers=args.num_workers)  # -> Dataset
    ds = ds.train_test_split(test_size=args.val, seed=42)  # -> DatasetDict
    ds_train = ds["train"]
    ds_eval = ds["test"]
elif args.stage == "math":  # ЕСЛИ СТАДИЯ == math
    ds = build_math_stage(limit=args.limit_math, num_workers=args.num_workers)  # -> Dataset
    ds = ds.train_test_split(test_size=args.val, seed=42)  # -> DatasetDict
    ds_train = ds["train"]
    ds_eval = ds["test"]
elif args.stage == "vision":  # ЕСЛИ СТАДИЯ == vision
    ds = build_vision_stage(limit=args.limit_vision, num_workers=args.num_workers)  # -> Dataset
    ds = ds.train_test_split(test_size=args.val, seed=42)  # -> DatasetDict
    ds_train = ds["train"]
    ds_eval = ds["test"]

In [ ]:
from transformers import EarlyStoppingCallback

FastVisionModel.for_training(model) # Переключение на обучение

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = ds_train,
    eval_dataset = ds_eval,
    args = training_args,
)

early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience = 3,     # Сколько ПРОВЕРОК нужно ждать перед тем как остановить обучение (при этом loss не изменяется больше чем на <early_stopping_threshold>)
    early_stopping_threshold = 0.008,  # Устанавливает, на сколько максимум может изменится loss но обучение
                                        # все равно остановится. Например, если значение = 0.008, при этом loss был 1.08(9)268
                                        #                                                                    и стал 1.08(1)793
                                        # в промежутке <early_stopping_patience>, то обучение останавливается
)
trainer.add_callback(early_stopping_callback)

In [ ]:
trainer_stats = trainer.train()

## Сохранение модели
*~~Два варианта — в адаптерах и GGUF~~*\
Только адаптеры, сохранение напрямую в GGUF невозможно из-за несовпадения тензоров

### Merged checkpoint для следующей стадии (обязательно)

Слива LoRA в базу и сохранение полного чекпоинта. Его загружаем как базу на следующей стадии (см. Args).

In [ ]:
merged_dir = os.path.join(args.outdir, f"merged_after_{args.stage}")
model.save_pretrained_merged(
    merged_dir,
    tokenizer,
    save_method="merged_16bit",
)

### Запаковка в архив для скачивания
*Так-то тоже обязательно, иначе как загружать модель в будущем?*

In [ ]:
!zip -r "after_{args.stage}.zip" "/kaggle/working/merged_after_{args.stage}"
from IPython.display import FileLink
FileLink(f"after_{args.stage}.zip") # здесь должен быть путь, для которого создается ссылка. не работает в colab (по очень интересной причине)